# Dither Algorithm Explorer

Apply every dither algorithm in `tools/dither_applier.py` to one image and compare them side by side, including zoomed crops that reveal each algorithm's signature artifacts (Floyd–Steinberg worms, Atkinson contrast, Bayer crosshatch, blue-noise grain).

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# Make the repo's tools importable from notebooks/.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'tools')))
import halftone_common as hc

def demo_image(size=256):
    """A synthetic test image: smooth ramp + gradient disk, for when you
    don't have a photo handy. Replace with hc.load_image('path.jpg')."""
    y, x = np.mgrid[0:size, 0:size] / size
    ramp = x
    disk = np.clip(1.2 - 2.0 * np.hypot(x - 0.5, y - 0.5), 0, 1)
    g = np.clip(0.5 * ramp + 0.5 * disk, 0, 1)
    return np.stack([g, np.roll(g, 20, 0), 1 - g], axis=-1)

img = demo_image()
plt.figure(figsize=(4,4)); plt.imshow(img); plt.title('source'); plt.axis('off'); plt.show()


In [ ]:
import dither_applier as da
from blue_noise_generator import generate as gen_bn

gray = hc.to_gray(img)
bn = gen_bn(64, 'void-cluster', np.random.default_rng(0))  # blue-noise tile


## All algorithms at 2 levels (1-bit)

In [ ]:
def run(algo, levels=2):
    if algo == 'bayer':
        return da.ordered_dither(gray, levels, 8)
    if algo == 'blue-noise':
        return da.blue_noise_dither(gray, levels, bn)
    return da.error_diffusion(gray, levels, da.KERNELS[algo], serpentine=False)

algos = ['bayer', 'floyd', 'atkinson', 'jjn', 'sierra', 'stevenson-arce', 'blue-noise']
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
ax = ax.ravel()
ax[0].imshow(gray, cmap='gray'); ax[0].set_title('source'); ax[0].axis('off')
for a, name in zip(ax[1:], algos):
    a.imshow(run(name), cmap='gray', vmin=0, vmax=1)
    a.set_title(name); a.axis('off')
plt.tight_layout(); plt.show()

## Zoom in on the artifacts
A flat midtone gradient is the cruelest test — it exposes worms and structure.

In [ ]:
ramp = np.tile(np.linspace(0, 1, 256), (128, 1))
fig, ax = plt.subplots(2, 2, figsize=(12, 6))
for a, name in zip(ax.ravel(), ['bayer','floyd','atkinson','blue-noise']):
    if name=='bayer': d = da.ordered_dither(ramp, 2, 8)
    elif name=='blue-noise': d = da.blue_noise_dither(ramp, 2, bn)
    else: d = da.error_diffusion(ramp, 2, da.KERNELS[name], False)
    a.imshow(d, cmap='gray', vmin=0, vmax=1); a.set_title(name); a.axis('off')
plt.tight_layout(); plt.show()

## Serpentine vs raster (Floyd–Steinberg)
Serpentine scanning breaks up the directional worm bias.

In [ ]:
raster = da.error_diffusion(ramp, 2, da.KERNELS['floyd'], False)
serp   = da.error_diffusion(ramp, 2, da.KERNELS['floyd'], True)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(raster, cmap='gray'); ax[0].set_title('raster'); ax[0].axis('off')
ax[1].imshow(serp, cmap='gray'); ax[1].set_title('serpentine'); ax[1].axis('off')
plt.show()

## Levels sweep
More output levels = subtler dither, less pattern.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, L in zip(ax, [2,3,4,8]):
    a.imshow(da.error_diffusion(gray, L, da.KERNELS['floyd'], True), cmap='gray', vmin=0, vmax=1)
    a.set_title(f'{L} levels'); a.axis('off')
plt.show()